In [1]:
import os
import ipywidgets as widgets
from langchain_openai import ChatOpenAI
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import BaseMessage
from IPython.display import display, Markdown
from typing import List

print("✓ Bibliotecas importadas correctamente")

# ── Configuración del modelo ──────────────────────────────────────────────────
token      = os.environ["GITHUB_TOKEN"]
endpoint   = "https://models.github.ai/inference"
model_name = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    base_url=endpoint,
    api_key=token,
    model=model_name,
    temperature=0.1,
    streaming=True          # ← Activa el streaming
)

print("✓ Modelo configurado con streaming")

# ── Prompt del sistema ────────────────────────────────────────────────────────
SYSTEM_PROMPT = """
Eres un asistente virtual interno de Invermar S.A., especializado en
remuneraciones y beneficios para los colaboradores del grupo Invermar
(Invermar S.A., Pesquera La Portada S.A., La Península S.A., Astilleros Calbuco S.A.).

Tus funciones principales son:
- Responder dudas sobre liquidaciones de sueldo, descuentos y haberes.
- Informar sobre beneficios vigentes (aguinaldos, bonos, CCAF, etc.).
- Orientar sobre vacaciones legales y progresivas (Art. 68 Código del Trabajo).
- Explicar el proceso y cálculo de finiquitos (Art. 172 CT).
- Aclarar dudas sobre AFP, Isapre/Fonasa e Impuesto Único (IUSC).
- Informar sobre normativa laboral chilena vigente.

Reglas importantes:
- Responde siempre en español, de forma cercana, cálida y amigable,
  como si fueras un compañero de trabajo que conoce muy bien el tema.
- Usa emojis ocasionalmente para hacer las respuestas más amenas,
  por ejemplo: 📄 para documentos, 💰 para temas de sueldo,
  🎁 para beneficios, 📅 para fechas, ✅ para confirmaciones,
  🐟 como símbolo de Invermar.
- No abuses de los emojis, usa máximo 2 o 3 por respuesta.
- Usa formato Markdown en tus respuestas: negritas, listas y títulos
  cuando corresponda para que se vea ordenado.
- Si no tienes certeza de un dato específico de la empresa, indícalo y sugiere
  consultar directamente con el área de Remuneraciones.
- No inventes cifras ni montos específicos si no los conoces con certeza.
- Sé conciso pero completo en tus respuestas.
"""

# ── Memory Window ─────────────────────────────────────────────────────────────
MEMORY_WINDOW = 6  # Número de mensajes a recordar (3 del usuario + 3 del bot)

class WindowChatMessageHistory(InMemoryChatMessageHistory):
    """Historial con ventana deslizante: solo guarda los últimos N mensajes."""

    max_messages: int = MEMORY_WINDOW

    def add_message(self, message: BaseMessage) -> None:
        super().add_message(message)
        # Si supera el límite, eliminar los más antiguos
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]

print(f"✓ Memory Window configurada: últimos {MEMORY_WINDOW} mensajes")

# ── Configurar cadena con memoria ─────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = WindowChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

print("✓ Memoria configurada correctamente")

# ── Interfaz visual con widgets ───────────────────────────────────────────────
session_id = "invermar_sesion"

# Encabezado
display(Markdown("---"))
display(Markdown("## 🐟 Chatbot Invermar — Remuneraciones y Beneficios"))
display(Markdown(f"Escribe tu pregunta y presiona **Enviar**. *(Memoria: últimas {MEMORY_WINDOW // 2} interacciones)*"))
display(Markdown("---"))

# Componentes visuales
entrada = widgets.Text(
    placeholder="Escribe tu pregunta aquí...",
    layout=widgets.Layout(width="70%")
)

boton = widgets.Button(
    description="Enviar 📨",
    button_style="primary",
    layout=widgets.Layout(width="15%")
)

boton_limpiar = widgets.Button(
    description="Limpiar 🗑️",
    button_style="warning",
    layout=widgets.Layout(width="12%")
)

salida = widgets.Output()

# ── Lógica del botón Enviar con streaming ────────────────────────────────────
def al_enviar(b):
    mensaje = entrada.value.strip()
    if not mensaje:
        return
    entrada.value = ""
    boton.disabled = True          # Deshabilitar mientras responde

    # Mostrar pregunta del usuario
    with salida:
        display(Markdown(f"**🧑 Tú:** {mensaje}"))
        display(Markdown("**🤖 Chatbot:** *escribiendo...*"))

    # Acumular respuesta completa via streaming
    respuesta_completa = ""
    for chunk in conversation.stream(
        {"input": mensaje},
        config={"configurable": {"session_id": session_id}}
    ):
        respuesta_completa += chunk.content

    # Mostrar historial completo actualizado
    with salida:
        salida.clear_output(wait=True)
        historial = get_session_history(session_id).messages
        for msg in historial:
            if msg.type == "human":
                display(Markdown(f"**🧑 Tú:** {msg.content}"))
            else:
                display(Markdown(f"**🤖 Chatbot:** {msg.content}"))
            display(Markdown("---"))

    boton.disabled = False         # Rehabilitar botón

# ── Lógica del botón Limpiar ──────────────────────────────────────────────────
def al_limpiar(b):
    store.clear()
    salida.clear_output()
    entrada.value = ""
    with salida:
        display(Markdown("*Conversación reiniciada. ¡Hola de nuevo! 👋*"))

boton.on_click(al_enviar)
boton_limpiar.on_click(al_limpiar)

# ── Mostrar interfaz ──────────────────────────────────────────────────────────
display(widgets.HBox([entrada, boton, boton_limpiar]))
display(salida)


✓ Bibliotecas importadas correctamente
✓ Modelo configurado con streaming
✓ Memory Window configurada: últimos 6 mensajes
✓ Memoria configurada correctamente


---

## 🐟 Chatbot Invermar — Remuneraciones y Beneficios

Escribe tu pregunta y presiona **Enviar**. *(Memoria: últimas 3 interacciones)*

---

Output()